# Validate the Premier League centre-back template

This notebook validates the two-stage averaging method: equal-weighted eligible appearances form each player's profile, then qualifying player profiles receive equal weight in the league template. Players need at least five eligible centre-back appearances to define the template.

In [ ]:
from pathlib import Path

import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mplsoccer import VerticalPitch

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (root for root in search_roots if (root / 'data' / 'processed').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')

PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT

In [ ]:
appearance_profiles = pd.read_parquet(
    PROCESSED_ROOT / 'centre_back_appearance_zone_profiles.parquet'
)
player_profiles = pd.read_parquet(
    PROCESSED_ROOT / 'centre_back_player_zone_profiles.parquet'
)
league_templates = pd.read_parquet(
    PROCESSED_ROOT / 'centre_back_league_templates.parquet'
)
quality_checks = pd.read_csv(
    PROCESSED_ROOT / 'centre_back_template_quality_checks.csv'
)

quality_checks

## Player inclusion and weighting

In [ ]:
player_summary = (
    player_profiles[[
        'player_id',
        'player_name',
        'teams',
        'eligible_appearances',
        'total_actions',
        'mean_actions_per_appearance',
        'total_minutes',
        'template_eligible',
        'minimum_template_appearances',
    ]]
    .drop_duplicates()
    .sort_values(['template_eligible', 'eligible_appearances'], ascending=[False, False])
    .reset_index(drop=True)
)

print(f'All centre-backs with profiles: {len(player_summary):,}')
print(f"Players defining the template: {player_summary['template_eligible'].sum():,}")
player_summary

In [ ]:
appearance_share_sums = appearance_profiles.groupby(
    ['match_id', 'team_id', 'player_id', 'layout']
)['action_share'].sum()
player_share_sums = player_profiles.groupby(
    ['player_id', 'layout']
)['mean_action_share'].sum()
template_share_sums = league_templates.groupby(
    'layout'
)['mean_action_share'].sum()

assert np.isclose(appearance_share_sums, 1).all()
assert np.isclose(player_share_sums, 1).all()
assert np.isclose(template_share_sums, 1).all()
assert league_templates['player_count'].eq(78).all()

pd.DataFrame(
    {
        'layout': template_share_sums.index,
        'template_share_sum': template_share_sums.values,
        'equal_weighted_players': [78, 78],
    }
)

## League template tables

In [ ]:
league_templates.sort_values(['layout', 'zone_id']).reset_index(drop=True)

## Visualise the first centre-back template

In [ ]:
pitch = VerticalPitch(
    pitch_type='opta',
    pitch_color='#22312b',
    line_color='white',
    line_zorder=2,
)
fig, axes = pitch.draw(nrows=1, ncols=2, figsize=(12, 8))
fig.patch.set_facecolor('#22312b')
shared_vmax = league_templates['mean_action_share'].max()
label_effects = [
    path_effects.Stroke(linewidth=2.5, foreground='black'),
    path_effects.Normal(),
]

for axis, layout in zip(axes, ['positional_20', 'traditional_18']):
    profile = (
        league_templates.loc[league_templates['layout'].eq(layout)]
        .sort_values('zone_id')
    )
    if layout == 'positional_20':
        bins = pitch.bin_statistic_positional(
            profile['center_x'],
            profile['center_y'],
            values=profile['mean_action_share'],
            statistic='sum',
            positional='full',
        )
        pitch.heatmap_positional(
            bins,
            ax=axis,
            cmap='cividis',
            edgecolors='#22312b',
            vmin=0,
            vmax=shared_vmax,
        )
        title = 'Positional 20-zone template'
    else:
        bins = pitch.bin_statistic(
            profile['center_x'],
            profile['center_y'],
            values=profile['mean_action_share'],
            statistic='sum',
            bins=(6, 3),
        )
        pitch.heatmap(
            bins,
            ax=axis,
            cmap='cividis',
            edgecolors='#22312b',
            vmin=0,
            vmax=shared_vmax,
        )
        title = 'Traditional 18-zone template'

    for zone in profile.itertuples(index=False):
        pitch.annotate(
            f'Z{zone.zone_id:02d}\n{zone.mean_action_share:.1%}',
            xy=(zone.center_x, zone.center_y),
            ax=axis,
            color='white',
            fontsize=8.5,
            fontweight='bold',
            ha='center',
            va='center',
            path_effects=label_effects,
            zorder=4,
        )
    axis.set_title(title, color='white', fontsize=14, pad=10)

fig.suptitle(
    'Premier League centre-back direct-action template',
    color='white',
    fontsize=17,
    y=0.98,
)
fig.text(
    0.5,
    0.94,
    'Equal-weighted mean of 78 player profiles; minimum five eligible appearances',
    color='#d9e2df',
    fontsize=11,
    ha='center',
)
fig.subplots_adjust(top=0.88, bottom=0.05, wspace=0.08)
plt.show()

## Forest centre-back profiles available for comparison

In [ ]:
forest_centre_backs = player_summary.loc[
    player_summary['teams'].str.contains('Forest', case=False, na=False)
].reset_index(drop=True)
forest_centre_backs